# cli

> Installed console commands: `segpt-train` and `segpt-submit`.

This module wires the library into two shell commands that work after `pip install -e .` or `uv sync`.

## Why two commands?

| Command | What it does |
|---|---|
| `segpt-train` | Runs the Hydra training pipeline (`scripts/train.py`) |
| `segpt-submit` | Submits a training job to a remote HPC cluster via LSF `bsub` |

Both read settings from the same `train.yaml` config file, so there is one source of truth.

## Design rule

Each function in this module does **exactly one thing**.
We develop each piece as raw code first, verify it works, then wrap it.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
#| hide
from nbdev.showdoc import *
import os

In [ ]:
#| export
from __future__ import annotations

import argparse
import runpy
import subprocess
import sys
from pathlib import Path

---
## Part 1 — `segpt-train`

### Problem: Hydra wants `--config-path` + `--config-name`, users want a plain file path

Hydra splits config location into two flags:
```
--config-path /abs/path/to/dir  --config-name train
```

We want users to just type:
```
segpt-train /any/path/to/train.yaml
```

So we intercept `sys.argv`, detect a `.yaml` argument, and rewrite it into what Hydra expects.
Any extra `key=value` overrides are passed through unchanged.

In [ ]:
# Test the argv rewrite logic in isolation — no Hydra needed

fake_argv = ['segpt-train', 'configs/train.yaml', 'train.epochs=80']
print(f'Fake argv: {fake_argv}')
print(f'Expand user: {Path(fake_argv[1]).expanduser()}')
print(f'Resolve: {Path(fake_argv[1]).expanduser().resolve()}')
cfg_file = Path(fake_argv[1]).expanduser().resolve()
cfg_file


In [ ]:

rewritten = [
    fake_argv[0],
    '--config-path', str(cfg_file.parent),
    '--config-name', cfg_file.stem,
    *fake_argv[2:],
]
print(rewritten)
# Expected: ['segpt-train', '--config-path', '<abs parent>', '--config-name', 'train', 'train.epochs=80']

In [ ]:
#| export
def _rewrite_argv_for_hydra() -> None:
    """If sys.argv[1] is a .yaml path, rewrite it to Hydra's --config-path / --config-name flags.

    Called once at the start of `segpt_train()`. Mutates `sys.argv` in place.
    """
    argv = sys.argv[1:]
    if not argv:
        return
    if argv[0].startswith("-") or not argv[0].endswith((".yml", ".yaml")):
        return  # already in Hydra format or no config supplied

    cfg_file = Path(argv[0]).expanduser().resolve()
    if not cfg_file.exists():
        raise FileNotFoundError(f"Config file not found: {cfg_file}")
    if not cfg_file.is_file():
        raise ValueError(f"Expected a config file, got a directory: {cfg_file}")

    sys.argv = [
        sys.argv[0],
        "--config-path", str(cfg_file.parent),
        "--config-name", cfg_file.stem,
        *argv[1:],
    ]

In [ ]:
path = Path('/path/to/segmentation_tools_pt')
path.exists()

In [ ]:
current_path = os.chdir(
    path
)

In [ ]:
# Test: verify the function works with a real path on this machine
import sys
from pathlib import Path

orig = sys.argv.copy()
print(f'Current orig: {orig}')
sys.argv = ['segpt-train', str(Path('configs/train.yaml').resolve()), 'train.epochs=5']
_rewrite_argv_for_hydra()
print(f'After rewrite: {sys.argv}')
assert '--config-path' in sys.argv
assert '--config-name' in sys.argv
assert sys.argv[sys.argv.index('--config-name') + 1] == 'train'
assert 'train.epochs=5' in sys.argv
print('argv rewrite OK:', sys.argv)
sys.argv = orig  # restore

In [ ]:
sys.argv

In [ ]:
#| export
def _find_repo_root() -> Path:
    """Locate the repository root — works in both notebook and installed-module context.

    Why two strategies?
    - ``__file__`` is defined when cli.py is imported as a module -> fast, reliable.
    - ``__file__`` is NOT defined in a Jupyter notebook cell -> walk up from cwd instead.

    The cwd fallback finds the first parent that contains both ``pyproject.toml``
    and a ``scripts/`` folder, which uniquely identifies this repo.
    """
    # Strategy 1: running as an installed module
    try:
        return Path(__file__).resolve().parents[2]
    except NameError:
        pass

    # Strategy 2: running inside a Jupyter notebook
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "scripts").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the repository root. "
        "Run this notebook from inside the cloned repository."
    )


In [ ]:
_find_repo_root()

In [ ]:
#| export

def _find_train_script() -> Path:
    """Return the absolute path to ``scripts/train.py``.

    Works in both notebook context and as an installed CLI command.
    """
    script = _find_repo_root() / "scripts" / "train.py"
    if not script.exists():
        raise FileNotFoundError(
            f"Training script not found at {script}.\n"
            "Make sure the package was installed from the repository (pip install -e .)."
        )
    return script


def _find_build_script() -> Path:
    """Return the absolute path to ``scripts/build.py``."""
    script = _find_repo_root() / "scripts" / "build.py"
    if not script.exists():
        raise FileNotFoundError(
            f"Patch-build script not found at {script}.\n"
            "Make sure the package was installed from the repository (pip install -e .)."
        )
    return script

In [ ]:
_find_train_script()

In [ ]:
#| test
# the script should exist when running from inside the repo
# (This will only pass on a machine where the package is installed editable)
try:
    p = _find_train_script()
    print('Found:', p)
except FileNotFoundError as e:
    print('Not found (expected outside repo):', e)

In [ ]:
#| export
def _run_hydra_script(script: Path, *forced_overrides: str) -> None:
    """Rewrite argv for Hydra, append forced overrides, and run a Hydra entry script."""
    _rewrite_argv_for_hydra()
    if forced_overrides:
        sys.argv = [*sys.argv, *forced_overrides]
    runpy.run_path(str(script), run_name="__main__")


def _run_train_script(*forced_overrides: str) -> None:
    """Rewrite argv for Hydra, append forced overrides, and run scripts/train.py."""
    _run_hydra_script(_find_train_script(), *forced_overrides)


def _run_build_script(*forced_overrides: str) -> None:
    """Rewrite argv for Hydra, append forced overrides, and run scripts/build.py."""
    _run_hydra_script(_find_build_script(), *forced_overrides)


def _is_build_job(config_path: Path, train_overrides: list[str], build_only_flag: bool) -> bool:
    """Return True when the submitted job should run segpt-build."""
    if build_only_flag or config_path.stem.startswith("build"):
        return True
    return any(ov.strip().lower() == "data.build_only=true" for ov in train_overrides)


def segpt_train() -> None:
    """Entry point for the `segpt-train` command.


    Accepts a plain config file path and rewrites it to Hydra format,
    then delegates to `scripts/train.py` via `runpy`.

    Example
    -------
        segpt-train configs/train.yaml
        segpt-train configs/train.yaml train.epochs=80 metrics.best_metric=recall
    """
    _run_train_script()


def segpt_build() -> None:
    """Entry point for the `segpt-build` command.


    Runs ``scripts/build.py`` with a slim build config (``configs/build.yaml``).
    ``configs/train.yaml`` still works for backward compatibility.

    Example
    -------
        segpt-build configs/build.yaml
        segpt-build configs/build.yaml data.force_rebuild=true patch.patch_size=512
    """
    _run_build_script()

In [ ]:
segpt_train()

---
## Part 2 — `segpt-submit`

### Problem statement

Training on a shared HPC cluster requires submitting a `bsub` job with site-specific flags.
We do not want those flags hardcoded in a shell script — they should live in the same
`train.yaml` config that drives training, under an `hpc:` section.

The `segpt-submit` command:
1. Reads `hpc:` settings from the YAML
2. Applies any `hpc.*=value` CLI overrides
3. Builds the exact `bsub` command for the chosen queue
4. Generates the inner bash job script that calls `segpt-train`
5. Submits (or prints with `--dry-run`)

### Example bsub patterns

**`gpu` queue** — shared nodes, filter by minimum GPU memory:
```bash
bsub -Is -q gpu \
     -gpu "num=3:j_exclusive=yes:gmem=24G" \
     -R 'select[ui==<your_resource_group> && osrel>=70 && type==X64LIN] rusage[mem=200G]' \
     -n 8 \
     -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err
```

**`gpu_compute` queue** — dedicated big-GPU nodes (40G+), pin to specific host:
```bash
bsub -Is -q gpu_compute \
     -gpu "num=1:j_exclusive=yes" \
     -R 'select[ui==<your_resource_group> && osrel>=70 && type==X64LIN] rusage[mem=200G]' \
     -m gpu-node-01 \
     -n 8 \
     -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err
```

Key differences from generic LSF:
- Memory goes in `rusage[mem=...G]` inside `-R`, **not** in `-M`
- GPU exclusivity is `j_exclusive=yes`, not `mode=exclusive_process`
- The resource `select` expression contains site-specific fields (`ui==<your_resource_group>`, `osrel`, `type`)

### Step 2a — Read `hpc:` values from YAML

Hydra config files are YAML. We use `pyyaml` if available, with a minimal regex fallback
so the CLI works even if only stdlib is installed.

In [ ]:
# Explore: what does pyyaml give us from train.yaml?
from pathlib import Path
import yaml

cfg_path = Path('/path/to/segmentation_tools_pt/configs/train.yaml')
data = yaml.safe_load(cfg_path.read_text())
print('hpc section:', data.get('hpc', {}))

In [ ]:
# Explore: applying a CLI override like hpc.queue=gpu_compute
hpc = dict(data.get('hpc', {}))
override = 'hpc.queue=gpu_compute'
key, val = override[4:].split('=', 1)  # strip 'hpc.' prefix
hpc[key] = val
print('queue after override:', hpc['queue'])  # gpu_compute

In [ ]:
#| export
def _load_yaml(config_path: Path) -> dict:
    """Load a YAML file into a dict. Uses pyyaml if available, falls back to regex parser."""
    import re
    try:
        import yaml
        return yaml.safe_load(config_path.read_text()) or {}
    except ImportError:
        pass

    # Minimal fallback: parses top-level sections and two-space-indented key: value lines
    data: dict = {}
    section = None
    for line in config_path.read_text().splitlines():
        sec = re.match(r'^(\w+)\s*:$', line)
        kv  = re.match(r'^\s{2}(\w+)\s*:\s*(.*)$', line)
        if sec:
            section = sec.group(1)
            data.setdefault(section, {})
        elif kv and section:
            val = kv.group(2).strip().strip('"').strip("'")
            data[section][kv.group(1)] = val
    return data

In [ ]:
_load_yaml(cfg_path)

In [ ]:
#| export
def _apply_hpc_overrides(hpc: dict, overrides: list[str]) -> dict:
    """Apply `hpc.key=value` CLI overrides to the hpc dict. Returns a new dict."""
    result = dict(hpc)
    for ov in overrides:
        if ov.startswith("hpc.") and "=" in ov:
            k, v = ov[4:].split("=", 1)
            result[k] = v
    return result

In [ ]:
# Test both functions together
raw = _load_yaml(Path('/path/to/segmentation_tools_pt/configs/train.yaml'))
hpc = raw.get('hpc', {})
hpc = _apply_hpc_overrides(hpc, ['hpc.queue=gpu_compute', 'hpc.host=gpu-node-01', 'train.epochs=80'])

assert hpc['queue'] == 'gpu_compute'
assert hpc['host'] == 'gpu-node-01'
assert 'train.epochs=80' not in hpc  # non-hpc overrides are NOT applied here
print('hpc after overrides:', hpc)

In [ ]:
#| export
def _parse_yaml_hpc(config_path: Path, overrides: list[str]) -> dict:
    """Load the `hpc:` section from a YAML config and apply CLI overrides.

    Example
    -------
        hpc = _parse_yaml_hpc(Path('configs/train.yaml'), ['hpc.queue=gpu_compute'])
        hpc['queue']  # 'gpu_compute'
    """
    data = _load_yaml(config_path)
    hpc  = dict(data.get("hpc", {}))
    return _apply_hpc_overrides(hpc, overrides)

### Step 2b — Build the GPU spec string

The `-gpu` flag in LSF has the form `num=N:j_exclusive=yes[:gmem=XG]`.
The `gmem` part is only added on the `gpu` queue (shared pool). On `gpu_compute` (dedicated big nodes)
you omit it because the node already has enough memory.

In [ ]:
# Explore: build GPU spec step by step
num_gpus    = 1
j_exclusive = True
gpu_mem_gb  = 24  # GB

spec = f"num={num_gpus}"
spec += ":j_exclusive=yes" if j_exclusive else ":j_exclusive=no"
if gpu_mem_gb:
    spec += f":gmem={gpu_mem_gb}G"

print(spec)  # num=3:j_exclusive=yes:gmem=24G

In [ ]:
#| export
def _build_gpu_spec(hpc: dict) -> str:
    """Build the LSF `-gpu` spec string.

    Example
    -------
        _build_gpu_spec({'num_gpus': 3, 'j_exclusive': True, 'gpu_mem_gb': 24})
        # 'num=3:j_exclusive=yes:gmem=24G'
        _build_gpu_spec({'num_gpus': 1, 'j_exclusive': True, 'gpu_mem_gb': ''})  # gpu_compute
        # 'num=1:j_exclusive=yes'
    """
    num_gpus    = hpc.get("num_gpus", 1)
    j_exclusive = str(hpc.get("j_exclusive", "true")).lower() in ("true", "yes", "1")
    gpu_mem_gb  = hpc.get("gpu_mem_gb", "")

    spec  = f"num={num_gpus}"
    spec += ":j_exclusive=yes" if j_exclusive else ":j_exclusive=no"
    if gpu_mem_gb and str(gpu_mem_gb) not in ("", "null", "None", "0"):
        spec += f":gmem={gpu_mem_gb}G"
    return spec

In [ ]:
# Test all combinations
assert _build_gpu_spec({'num_gpus': 3, 'j_exclusive': True,  'gpu_mem_gb': 24}) == 'num=3:j_exclusive=yes:gmem=24G'
assert _build_gpu_spec({'num_gpus': 1, 'j_exclusive': True,  'gpu_mem_gb': ''}) == 'num=1:j_exclusive=yes'
assert _build_gpu_spec({'num_gpus': 1, 'j_exclusive': False, 'gpu_mem_gb': 40}) == 'num=1:j_exclusive=no:gmem=40G'
print('GPU spec tests passed')

### Step 2c — Build the `-R` resource expression

In LSF, memory goes into `rusage[mem=...G]` inside the `-R` string, **not** in `-M`.
The `select[...]` part filters hosts by site-specific attributes.

In [ ]:
# Explore: build -R string
resource_select = "ui==<your_resource_group> && osrel>=70 && type==X64LIN"
mem_gb = 200
r_expr = f"select[{resource_select}] rusage[mem={mem_gb}G]"
print(r_expr)

In [ ]:
#| export
def _build_resource_expr(hpc: dict) -> str:
    """Build the LSF `-R` resource expression.

    Memory lives in `rusage[mem=...G]` (not in `-M`).
    The `select[...]` clause is site-specific and configurable via `hpc.resource_select`.

    Example
    -------
        _build_resource_expr({'mem_gb': 200})
        # "select[ui==<your_resource_group> && osrel>=70 && type==X64LIN] rusage[mem=200G]"
    """
    select = str(hpc.get(
        "resource_select",
        "ui==<your_resource_group> && osrel>=70 && type==X64LIN",
    ))
    mem_gb = hpc.get("mem_gb", 200)
    return f"select[{select}] rusage[mem={mem_gb}G]"

In [ ]:
# Test
expr = _build_resource_expr({'mem_gb': 200})
assert 'rusage[mem=200G]' in expr
assert 'your_resource_group' in expr

# Custom select
expr2 = _build_resource_expr({'mem_gb': 64, 'resource_select': 'type==X64LIN'})
assert expr2 == 'select[type==X64LIN] rusage[mem=64G]'
print('Resource expr tests passed')

### Step 2d — Assemble the full bsub command list

In [ ]:
# Explore: assemble bsub for gpu queue
hpc_gpu = {
    'job_name': 'segpt_train', 'queue': 'gpu', 'device': 'gpu',
    'interactive': True, 'num_gpus': 3, 'j_exclusive': True, 'gpu_mem_gb': 24,
    'host': '', 'num_cpus': 8, 'mem_gb': 200, 'project': ''
}

cmd = ['bsub', '-Is', '-q', 'gpu', '-n', '8',
       '-gpu', _build_gpu_spec(hpc_gpu),
       '-R', _build_resource_expr(hpc_gpu),
       '-o', 'logs/segpt_train_%J.out', '-e', 'logs/segpt_train_%J.err']
print(' '.join(f"'{c}'" if ' ' in c else c for c in cmd))

In [ ]:
#| export
def _build_bsub_cmd(hpc: dict) -> list[str]:
    """Assemble the complete bsub command argument list from the hpc config dict.

    GPU queue example:
        bsub -Is -q gpu -n 8
             -gpu 'num=3:j_exclusive=yes:gmem=24G'
             -R 'select[ui==<your_resource_group>...] rusage[mem=200G]'
             -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err

    gpu_compute queue (big node, pinned host):
        bsub -Is -q gpu_compute -n 8
             -gpu 'num=1:j_exclusive=yes'
             -R 'select[...] rusage[mem=200G]'
             -m gpu-node-01
             -o logs/segpt_train_%J.out -e logs/segpt_train_%J.err
    """
    job_name    = str(hpc.get("job_name", "segpt_train"))
    queue       = str(hpc.get("queue", "gpu"))
    device      = str(hpc.get("device", "gpu")).lower()
    num_cpus    = hpc.get("num_cpus", 8)
    interactive = str(hpc.get("interactive", "true")).lower() in ("true", "yes", "1")
    host        = str(hpc.get("host", ""))
    project     = str(hpc.get("project", ""))

    cmd = ["bsub"]
    if interactive:
        cmd += ["-Is"]
    cmd += ["-q", queue, "-n", str(num_cpus)]

    if device == "gpu":
        cmd += ["-gpu", _build_gpu_spec(hpc)]

    cmd += ["-R", _build_resource_expr(hpc)]

    if host and host not in ("", "null", "None"):
        cmd += ["-m", host]
    if project and project not in ("", "null", "None"):
        cmd += ["-P", project]

    cmd += ["-o", f"logs/{job_name}_%J.out", "-e", f"logs/{job_name}_%J.err"]
    return cmd

In [ ]:
# Test: gpu queue
cmd = _build_bsub_cmd(hpc_gpu)
assert cmd[0] == 'bsub'
assert '-Is' in cmd
assert '-gpu' in cmd
assert 'num=3:j_exclusive=yes:gmem=24G' in cmd
assert '-m' not in cmd  # no host pinning on shared queue
print('GPU queue cmd:', ' '.join(cmd))

# Test: gpu_compute queue (big node, no gmem, pin host)
hpc_big = {**hpc_gpu, 'queue': 'gpu_compute', 'num_gpus': 1, 'gpu_mem_gb': '', 'host': 'gpu-node-01'}
cmd2 = _build_bsub_cmd(hpc_big)
assert 'gpu-node-01' in cmd2
assert 'gmem' not in ' '.join(cmd2)
print('gpu_compute cmd:', ' '.join(cmd2))

# Test: CPU queue (no -gpu flag)
hpc_cpu = {**hpc_gpu, 'device': 'cpu', 'queue': 'normal'}
cmd3 = _build_bsub_cmd(hpc_cpu)
assert '-gpu' not in cmd3
print('CPU cmd:', ' '.join(cmd3))

### Step 2e — Generate the inner bash job script

This is the script that LSF runs on the compute node.
It activates the environment, syncs deps, and calls `segpt-train`.

In [ ]:
# Explore: what should the job script look like?
hpc_ex = {'device': 'gpu', 'env_activate': 'source .venv/bin/activate',
           'repo_dir': '/path/to/segmentation_tools_pt'}
config_path_ex = '/path/to/segmentation_tools_pt/configs/train.yaml'
train_overrides_ex = ['train.epochs=80']

train_cmd = f'segpt-train "{config_path_ex}"'
if train_overrides_ex:
    train_cmd += ' ' + ' '.join(train_overrides_ex)

script = f"""#!/usr/bin/env bash
set -euo pipefail
echo "=== Job ${{LSB_JOBID}} | $(hostname) | $(date) ==="
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

{hpc_ex['env_activate']}

cd "{hpc_ex['repo_dir']}"
uv sync --quiet

{train_cmd}

echo "=== Done at $(date) ==="
"""
print(script)

In [ ]:
#| export
def _build_job_script(
    hpc: dict,
    config_path: Path,
    train_overrides: list[str],
    build_job: bool = False,
) -> str:
    """Generate the bash script that runs inside the LSF job.

    Minimal script: header → GPU info → cd to repo → segpt-train or segpt-build.
    No .bashrc sourcing — the LSF submission environment already has the
    correct PATH (segpt-train visible via the activated venv).
    """
    device   = str(hpc.get("device", "gpu")).lower()
    repo_dir = str(hpc.get("repo_dir", "/path/to/segmentation_tools_pt"))

    cmd_name  = "segpt-build" if build_job else "segpt-train"
    run_cmd   = f'{cmd_name} "{config_path}"'
    if train_overrides:
        run_cmd += " " + " ".join(train_overrides)

    gpu_block = (
        'nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv,noheader'
        if device == "gpu" else ""
    )

    step_label = "patch build" if build_job else "training"

    return f"""#!/usr/bin/env bash
set -euo pipefail

echo "=== Job ${{LSB_JOBID:-local}} | $(hostname) | $(date) ==="
{gpu_block}

echo "[step] entering repo..."
if [[ ! -d "{repo_dir}" ]]; then
    echo "[ERROR] repo_dir does not exist: {repo_dir}"
    echo "[ERROR] check hpc.repo_dir in your train.yaml"
    exit 1
fi
cd "{repo_dir}"
echo "[ok]   cwd=$(pwd)"
echo "[ok]   {cmd_name}=$(which {cmd_name})"

echo "[step] starting {step_label}..."
echo "============================================================"
{run_cmd}
echo "============================================================"
echo "=== Done at $(date) ==="
"""


In [ ]:
# Test
script = _build_job_script(
    {'device': 'gpu', 'env_activate': 'conda activate segpt', 'repo_dir': '$HOME/repo'},
    Path('/home/user/configs/train.yaml'),
    ['train.epochs=80'],
)
assert 'nvidia-smi' in script
assert 'train.epochs=80' in script
assert 'uv sync' in script

# CPU job should NOT have nvidia-smi
script_cpu = _build_job_script({'device': 'cpu', 'env_activate': '', 'repo_dir': '$HOME/repo'},
                                Path('/home/user/configs/train.yaml'), [])
assert 'nvidia-smi' not in script_cpu
print('Job script tests passed')

### Step 2f — Print a human-readable submission summary

In [ ]:
#| export
def _print_submit_summary(
    hpc: dict,
    config_path: Path,
    train_overrides: list[str],
    *,
    build_job: bool = False,
) -> None:
    """Print a summary of what will be submitted before calling bsub."""
    device  = str(hpc.get("device", "gpu")).lower()
    project = str(hpc.get("project", ""))
    build_only = build_job

    print("=" * 50)
    print(f"  Job      : {hpc.get('job_name', 'segpt_train')}")
    print(f"  Queue    : {hpc.get('queue', 'gpu')}  |  Device: {device}")
    print(f"  Mode     : {'build-only patch cache' if build_only else 'training'}")
    if device == "gpu":
        print(f"  GPUs     : {hpc.get('num_gpus', 1)}"
              f"  j_exclusive={hpc.get('j_exclusive', True)}"
              f"  gmem={hpc.get('gpu_mem_gb', 'any')}GB")
        host = str(hpc.get("host", ""))
        if host and host not in ("", "null", "None"):
            print(f"  Host     : {host}")
    print(f"  CPUs     : {hpc.get('num_cpus', 8)}"
          f"  |  Mem: {hpc.get('mem_gb', 200)}GB")
    if project and project not in ("", "null", "None"):
        print(f"  Project  : {project}")
    print(f"  Config   : {config_path}")
    if train_overrides:
        print(f"  Overrides: {' '.join(train_overrides)}")
    print("=" * 50)

In [ ]:
# Visual check
_print_submit_summary(
    {'job_name': 'segpt_train', 'queue': 'gpu', 'device': 'gpu',
     'num_gpus': 3, 'j_exclusive': True, 'gpu_mem_gb': 24,
     'num_cpus': 8, 'mem_gb': 200, 'project': ''},
    Path('/home/user/configs/train.yaml'),
    ['train.epochs=80'],
)

### Step 2g — Assemble into `segpt_submit()`

All the pieces are tested. Now we wire them together with `argparse`.

In [ ]:
#| export
def _seggpt_main_args():
    parser = argparse.ArgumentParser(
        prog="segpt-submit",
        description=(
            "Submit a segpt-train job to a remote HPC cluster (LSF bsub).\n\n"
            "All job settings are read from the hpc: section of your train.yaml.\n"
            "Override any field on the command line with key=value pairs.\n\n"
            "hpc.* pairs  → control bsub (queue, GPUs, memory, ...)\n"
            "other pairs  → forwarded to segpt-train (epochs, batch_size, ...)"
        ),
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=(
            "examples:\n"
            "  segpt-submit configs/train.yaml\n"
            "  segpt-submit configs/train.yaml --build-only\n"
            "  segpt-submit configs/train.yaml train.epochs=80 hpc.queue=gpu_compute hpc.host=gpu-node-01\n"
            "  segpt-submit configs/train.yaml --build-only hpc.queue=gpu_compute patch.patch_size=512\n"
            "  segpt-submit configs/train.yaml hpc.device=cpu hpc.queue=normal\n"
            "  segpt-submit configs/train.yaml --dry-run\n"
        ),
    )
    parser.add_argument(
        "config",
        help="Path to your Hydra train.yaml config file",
    )
    parser.add_argument(
        "overrides",
        nargs="*",
        metavar="key=value",
        help="Override config values. hpc.* pairs control bsub; others go to segpt-train.",
    )
    parser.add_argument(
        "--build-only",
        action="store_true",
        help="Submit a patch-build-only job by forcing data.build_only=true",
    )
    parser.add_argument(
        "--dry-run",
        action="store_true",
        help="Print the bsub command and job script without submitting",
    )
    return parser.parse_args()


In [ ]:
#| export
def segpt_submit() -> None:
    """Entry point for the `segpt-submit` command.


    Reads hpc: settings from a Hydra config file and submits
    a training or patch-build job to a remote HPC cluster via LSF bsub.


    hpc.* key=value pairs control job submission.
    All other key=value pairs are forwarded to segpt-train.


    Example
    -------
        segpt-submit configs/train.yaml
        segpt-submit configs/build.yaml
        segpt-submit configs/build.yaml --build-only
        segpt-submit configs/train.yaml train.epochs=80 hpc.queue=gpu_compute hpc.host=gpu-node-01
        segpt-submit configs/build.yaml hpc.queue=normal patch.patch_size=512
        segpt-submit configs/train.yaml hpc.device=cpu hpc.queue=normal
        segpt-submit configs/train.yaml --dry-run
    """
    args = _seggpt_main_args()
    print(f"[info] parsed args: {args}", flush=True)

    config_path = Path(args.config).expanduser().resolve()
    if not config_path.exists():
        print(f"[error] Config file not found: {config_path}", file=sys.stderr)
        sys.exit(1)

    overrides: list[str]       = args.overrides
    train_overrides: list[str] = [o for o in overrides if not o.startswith("hpc.")]
    build_job = _is_build_job(config_path, train_overrides, args.build_only)

    hpc      = _parse_yaml_hpc(config_path, overrides)
    bsub_cmd = _build_bsub_cmd(hpc)
    script   = _build_job_script(hpc, config_path, train_overrides, build_job=build_job)

    _print_submit_summary(hpc, config_path, train_overrides, build_job=build_job)

    if args.dry_run:
        print("\n[dry-run] bsub command:")
        print("  " + " ".join(bsub_cmd))
        print("\n[dry-run] job script:")
        print(script)
        return

    Path("logs").mkdir(exist_ok=True)
    interactive = str(hpc.get("interactive", "true")).lower() in ("true", "yes", "1")
    if interactive:
        # -Is: bsub blocks and streams the job output live to this terminal
        proc = subprocess.run(bsub_cmd, input=script, text=True)
    else:
        # fire-and-forget: just capture the job-ID line and return
        proc = subprocess.run(bsub_cmd, input=script, text=True, capture_output=True)
        if proc.stdout:
            print(proc.stdout.strip())
        if proc.returncode != 0:
            print(proc.stderr.strip(), file=sys.stderr)
    if proc.returncode != 0:
        sys.exit(proc.returncode)


---
## Full dry-run test

Simulate `segpt-submit configs/train.yaml train.epochs=80 --dry-run` without actually calling bsub.

In [ ]:
# End-to-end dry-run simulation
import sys
from pathlib import Path

config = Path('/path/to/segmentation_tools_pt/configs/train.yaml').resolve()


In [ ]:

overrides = ['train.epochs=80', 'hpc.queue=gpu', 'hpc.num_gpus=1', 'hpc.gpu_mem_gb=24']

overrides

In [ ]:

train_overrides = [o for o in overrides if not o.startswith('hpc.')]
train_overrides

In [ ]:

hpc = _parse_yaml_hpc(config, overrides)
hpc

In [ ]:


bsub_cmd = _build_bsub_cmd(hpc)
bsub_cmd

In [ ]:

script = _build_job_script(hpc, config, train_overrides)
print(script)

In [ ]:

_print_submit_summary(hpc, config, train_overrides)
print('\nbsub command:')
print(' ', ' '.join(bsub_cmd))
print('\njob script preview (first 10 lines):')
for line in script.splitlines()[:10]:
    print(' ', line)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()